In [13]:
import re
import shutil
import subprocess
import unicodedata
import urllib.request
from urllib.parse import urljoin
from pathlib import Path
import geopandas as gpd
import pandas as pd
from nume_zona import nume_zona
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "transform_data" else Path.cwd()

In [14]:
def normalize_uat_name(s: str) -> str:
    """Elimina diacriticele, cratimele si prefixul rangului administrativ (MUNICIPIUL/ORASUL/COMUNA)."""
    s = "".join(c for c in unicodedata.normalize("NFD", str(s)) if unicodedata.category(c) != "Mn")
    s = s.upper().replace("-", " ").strip()
    s = re.sub(r"^(MUNICIPIUL|ORASUL|ORAS|COMUNA)\s+", "", s)
    return re.sub(r"\s+", " ", s)

In [15]:
# venituri
DPFBL_PAGE_URL = "http://www.dpfbl.mdrap.ro/sit_ven_si_chelt_uat.html"

DPFBL_FALLBACK_URLS = {
    2025: "http://www.dpfbl.mdrap.ro/venituri_si_cheltuieli_uat/Anexa24_2025.xlsx",
    2024: "http://www.dpfbl.mdrap.ro/venituri_si_cheltuieli_uat/centra2024.xlsx",
    2023: "http://www.dpfbl.mdrap.ro/venituri_si_cheltuieli_uat/centra2023.xlsx",
    2022: "http://www.dpfbl.mdrap.ro/venituri_si_cheltuieli_uat/Centralizator2022.xlsx",
    2021: "http://www.dpfbl.mdrap.ro/venituri_si_cheltuieli_uat/Centralizator_2021.xlsx",
    **{
        y: f"http://www.dpfbl.mdrap.ro/venituri_si_cheltuieli_uat/Venituri_si_cheltuieli_uat_anul_{y}_(anexa_24).xlsx"
        for y in range(2015, 2021)
    },
    **{
        y: f"http://www.dpfbl.mdrap.ro/venituri_si_cheltuieli_uat/Venituri_si_cheltuieli_uat_anul_{y}_(anexa_24).xls"
        for y in range(2010, 2015)
    },
    **{
        y: f"http://www.dpfbl.mdrap.ro/venituri_si_cheltuieli_uat/Venituri_si_cheltuieli_uat_anul_{y}_(anexa24).xls"
        for y in range(2006, 2010)
    },
    **{
        y: f"http://www.dpfbl.mdrap.ro/venituri_si_cheltuieli_uat/Venituri_si_cheltuieli_uat_anul_{y}_(anexa7).xls"
        for y in range(1999, 2006)
    },
}


def gaseste_url_venituri(an: int = 2025) -> str:
    """Gaseste link-ul Excel de pe http://www.dpfbl.mdrap.ro/sit_ven_si_chelt_uat.html pentru anul cerut."""
    an = int(an)
    try:
        req = urllib.request.Request(DPFBL_PAGE_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=20) as resp:
            html = resp.read().decode("utf-8", errors="ignore")
        hrefs = re.findall(r'href=["\']([^"\']+\.xlsx?)["\']', html, flags=re.IGNORECASE)
        for href in hrefs:
            if re.search(rf"(?<!\d){an}(?!\d)", href):
                return urljoin(DPFBL_PAGE_URL, href)
    except Exception:
        pass

    if an in DPFBL_FALLBACK_URLS:
        return DPFBL_FALLBACK_URLS[an]
    raise ValueError(f"Nu s-a gasit fisier Excel pe {DPFBL_PAGE_URL} pentru anul {an}.")


def descarca_venituri(an: int = 2025) -> pd.DataFrame:
    """Descarca Excel-ul de venituri UAT pentru anul `an` si extrage UAT-urile judetului activ."""
    url = gaseste_url_venituri(an)
    ext = ".xls" if url.lower().endswith(".xls") else ".xlsx"
    venituri_path = PROJECT_ROOT / "raw_data" / f"venituri_per_uat{ext}"
    out_path = PROJECT_ROOT / "vizualizare" / "venituri_per_uat.csv"
    venituri_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=60) as resp:
        venituri_path.write_bytes(resp.read())

    nume_judet = nume_zona()
    judet_norm = normalize_uat_name(nume_judet)
    xl = pd.ExcelFile(venituri_path)

    if "Sheet2" in xl.sheet_names:
        venituri = pd.read_excel(venituri_path, sheet_name="Sheet2").iloc[5:, :2]
        venituri.columns = ["Cod fiscal - UAT", "Venituri totale"]
        venituri = venituri.dropna(subset=["Cod fiscal - UAT"]).reset_index(drop=True)

        judet_mask = venituri["Cod fiscal - UAT"].apply(normalize_uat_name).str.contains(
            rf"^JUDETUL:\s*{re.escape(judet_norm)}$"
        )
        judet_index = venituri.index[judet_mask][0]
        rest_venituri = venituri.iloc[judet_index + 1:]
        next_judet = rest_venituri.index[
            rest_venituri["Cod fiscal - UAT"].astype(str).str.contains("Judetul:", case=False)
        ]
        end_index = next_judet[0] if len(next_judet) > 0 else len(venituri)

        venituri_judet = venituri.iloc[judet_index + 8:end_index].reset_index(drop=True)
        venituri_judet = venituri_judet.loc[
            venituri_judet["Cod fiscal - UAT"].astype(str).str.contains("-")
        ].copy()
        venituri_judet["UAT"] = (
            venituri_judet["Cod fiscal - UAT"].str.split("-", n=1).str[1].apply(normalize_uat_name)
        )
        venituri_judet["Venituri totale"] = venituri_judet["Venituri totale"].astype(float).round(2)
        venituri_judet = venituri_judet.reset_index(drop=True)
    else:
        summary_sheet = "total TARA" if "total TARA" in xl.sheet_names else xl.sheet_names[0]
        summary_df = pd.read_excel(venituri_path, sheet_name=summary_sheet).iloc[:, :2].dropna()
        summary_df.columns = ["cod", "judet"]
        match = summary_df[summary_df["judet"].apply(normalize_uat_name).str.contains(judet_norm)]
        cod_judet = str(int(float(match["cod"].iloc[0])))
        target_sheet = next(
            s for s in xl.sheet_names if str(s).strip() == cod_judet or str(s).strip().startswith(f"{cod_judet} ")
        )
        df_j = pd.read_excel(venituri_path, sheet_name=target_sheet).iloc[:, :4]
        df_j.columns = ["crt", "SIRUTA", "UAT_raw", "Venituri totale"]
        df_j = df_j[pd.to_numeric(df_j["SIRUTA"], errors="coerce").notna() & df_j["UAT_raw"].notna()].copy()
        df_j["SIRUTA"] = df_j["SIRUTA"].astype(int).astype(str)

        uat_gpkg = gpd.read_file(PROJECT_ROOT / "raw_data" / "UAT.gpkg")[["natCode", "name"]].copy()
        uat_gpkg["SIRUTA"] = uat_gpkg["natCode"].astype(str)
        siruta_to_name = dict(zip(uat_gpkg["SIRUTA"], uat_gpkg["name"].apply(normalize_uat_name)))

        df_j["Cod fiscal - UAT"] = df_j["SIRUTA"] + " - " + df_j["UAT_raw"].astype(str).str.strip()
        df_j["UAT"] = df_j["SIRUTA"].map(siruta_to_name).fillna(df_j["UAT_raw"].apply(normalize_uat_name))
        df_j["Venituri totale"] = df_j["Venituri totale"].astype(float).round(2)
        venituri_judet = df_j[["Cod fiscal - UAT", "Venituri totale", "UAT"]].reset_index(drop=True)

    venituri_judet.to_csv(out_path)
    return venituri_judet


venituri_judet = descarca_venituri(an=2025)

In [4]:
venituri_judet

,Cod fiscal - UAT,Venituri totale,UAT
0,4230487 - MUNICIPIUL ORADEA,1.541970e+09,ORADEA
1,4348947 - MUNICIPIUL MARGHITA,7.572727e+07,MARGHITA
2,4593423 - MUNICIPIUL SALONTA,1.270301e+08,SALONTA
3,4794567 - MUNICIPIUL BEIUS,6.854700e+07,BEIUS
4,4348920 - ORASUL ALESD,1.278562e+08,ALESD
...,...,...,...
96,5518519 - COMUNA CABESTI,7.444332e+06,CABESTI
97,5518527 - COMUNA FINIS,2.062546e+07,FINIS
98,5543628 - COMUNA CAPALNA,7.758341e+06,CAPALNA
99,5628791 - COMUNA DOBRESTI,1.898264e+07,DOBRESTI


In [5]:
# populatia
def _gaseste_rscript() -> str:
    rscript = shutil.which("Rscript")
    if rscript:
        return rscript
    r_dir = Path(r"C:\Program Files\R")
    if r_dir.exists():
        candidati = sorted(r_dir.glob("R-*/bin/Rscript.exe"), reverse=True)
        if candidati:
            return str(candidati[0])
    raise FileNotFoundError("Nu a fost gasit Rscript.exe.")


def tempo_pop(an: int = 2025) -> pd.DataFrame:
    """Ruleaza transform_data/tempo_pop.r pentru anul `an` si extrage populatia UAT pentru judetul activ."""
    script_r = PROJECT_ROOT / "transform_data" / "tempo_pop.r"
    subprocess.run([_gaseste_rscript(), str(script_r), str(an)], check=True)

    pop_path = PROJECT_ROOT / "raw_data" / "populatie_per_uat.xlsx"
    out_path = PROJECT_ROOT / "vizualizare" / "populatie_per_uat.xlsx"
    out_path.parent.mkdir(parents=True, exist_ok=True)

    pop = pd.read_excel(pop_path)
    judet_norm = normalize_uat_name(nume_zona())

    if "Judet" in pop.columns and "SIRUTA_UAT" in pop.columns:
        pop = pop[pop["Judet"].apply(normalize_uat_name) == judet_norm].copy()
        pop = pop.rename(columns={"SIRUTA_UAT": "SIRUTA UAT"}).reset_index(drop=True)
        pop["Populatie"] = pop["Populatie"].astype(float)
        pop["SIRUTA"] = pop["SIRUTA"].astype(str)
        pop["UAT"] = pop["UAT"].apply(normalize_uat_name)
        pop = pop[["SIRUTA UAT", "Populatie", "SIRUTA", "UAT"]]
    else:
        pop = pop.iloc[5:106, 3:5].rename(columns={"Unnamed: 3": "SIRUTA UAT", "Unnamed: 4": "Populatie"})
        pop = pop.reset_index(drop=True)
        pop["Populatie"] = pop["Populatie"].astype(float)
        pop["SIRUTA"] = pop["SIRUTA UAT"].str.split(" ").str[0]
        pop["UAT"] = pop["SIRUTA UAT"].str.split(" ", n=1).str[1].apply(normalize_uat_name)

    pop.to_csv(out_path)
    return pop


pop = tempo_pop(an=2025)

In [6]:
pop.head(15)

,SIRUTA UAT,Populatie,SIRUTA,UAT
0,26564 MUNICIPIUL ORADEA,213622.0,26564,ORADEA
1,26804 MUNICIPIUL BEIUS,10447.0,26804,BEIUS
2,26877 MUNICIPIUL MARGHITA,16448.0,26877,MARGHITA
3,26975 MUNICIPIUL SALONTA,17931.0,26975,SALONTA
4,26699 ORAS ALESD,10927.0,26699,ALESD
5,26920 ORAS NUCET,2003.0,26920,NUCET
6,30915 ORAS SACUENI,12326.0,30915,SACUENI
7,26840 ORAS STEI,6375.0,26840,STEI
8,32027 ORAS VALEA LUI MIHAI,10360.0,32027,VALEA LUI MIHAI
9,27007 ORAS VASCAU,2088.0,27007,VASCAU


In [7]:
# venituri + populatie
venit_pop = venituri_judet.merge(pop, on="UAT", how="right")
venit_pop = venit_pop[["UAT", "SIRUTA","Venituri totale", "Populatie"]]

In [8]:
venit_pop["Venit per capita"] = (venit_pop["Venituri totale"] / venit_pop["Populatie"]).round(2)

In [9]:
venit_pop["Economie Punctaj"] = venit_pop["Venit per capita"].rank(pct=True) * 100

In [10]:
venit_pop.head(40)

,UAT,SIRUTA,Venituri totale,Populatie,Venit per capita,Economie Punctaj
0,ORADEA,26564,1.541970e+09,213622.0,7218.22,81.188119
1,BEIUS,26804,6.854700e+07,10447.0,6561.40,74.257426
2,MARGHITA,26877,7.572727e+07,16448.0,4604.04,42.574257
3,SALONTA,26975,1.270301e+08,17931.0,7084.39,80.198020
4,ALESD,26699,1.278562e+08,10927.0,11700.94,99.009901
5,NUCET,26920,1.933114e+07,2003.0,9651.09,94.059406
6,SACUENI,30915,5.042569e+07,12326.0,4091.00,29.702970
7,STEI,26840,7.983483e+07,6375.0,12523.11,100.000000
8,VALEA LUI MIHAI,32027,9.767497e+07,10360.0,9428.09,93.069307
9,VASCAU,27007,1.226367e+07,2088.0,5873.41,71.287129


In [11]:
# uat
uat_path = PROJECT_ROOT / "raw_data" / "UAT.GPKG"
uat = gpd.read_file(uat_path)
uat= uat.loc[:, ["natCode",  "geometry"]]
uat.rename(columns={"natCode": "SIRUTA"}, inplace=True)
uat["SIRUTA"] = uat["SIRUTA"].astype(str)

C:\Users\gblid\AppData\Local\Temp\ipykernel_18992\1302797961.py:5: Pandas4Warning: The inplace keyword in GeoDataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  uat.rename(columns={"natCode": "SIRUTA"}, inplace=True)


In [12]:
uat

,SIRUTA,geometry
0,153400,"POLYGON ((539844.121 245021.402, 539842.417 24..."
1,152314,"MULTIPOLYGON (((542412.565 245177.694, 542581...."
2,151978,"POLYGON ((535822.044 248653.332, 535731.377 24..."
3,151709,"MULTIPOLYGON (((495944.798 252581.861, 495950...."
4,155118,"POLYGON ((527174.86 252332.084, 529398.443 251..."
...,...,...
3181,60570,"POLYGON ((791944.91 293726.84, 791928.143 2937..."
3182,60534,"POLYGON ((791918.259 286886.895, 791901.52 286..."
3183,179711,"POLYGON ((561351.633 371155.804, 561302.438 37..."
3184,67256,"POLYGON ((564924.681 379635.636, 564934.392 37..."
